In [0]:
import yaml

with open("/Workspace/Users/roksolana.shendiu770@softserve.academy/petroleum-consumption-pipeline/lab4/prod/pipeline_config.yaml") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
bronze_schema = config["bronze_schema"]
silver_schema = config["silver_schema"]

source_table = config["consumption"]["source_table"]
target_table = config["consumption"]["target_table"]

bronze_full_name = f"{catalog}.{bronze_schema}.{source_table}"
silver_full_name = f"{catalog}.{silver_schema}.{target_table}"

In [0]:
import logging
from pyspark.sql.functions import (
    col, row_number, current_timestamp, sha2, concat_ws, expr, lit, lead
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

logger = logging.getLogger("silver_prices_pipeline")
logger.setLevel(logging.INFO)

try:
    bronze_df = spark.table(bronze_full_name)

    cleaned_df = (bronze_df
        .withColumnRenamed("series", "series_bk")
        .withColumnRenamed("product-name", "product_name")
        .withColumn("effective_from", expr("try_cast(period as date)"))
        .withColumn("price", expr("try_cast(value as decimal(10,3))"))
        .select(
            "series_bk", "product_name", "units", "price", "effective_from",
            "source_filename", "ingestion_timestamp"
        )
    )

    w_dedup = Window.partitionBy("series_bk", "effective_from") \
                     .orderBy(col("ingestion_timestamp").desc())

    deduped_df = (cleaned_df
        .withColumn("rn", row_number().over(w_dedup))
        .filter(col("rn") == 1)
        .drop("rn")
    )

    w_ordered = Window.partitionBy("series_bk").orderBy("effective_from")

    final_df = deduped_df.withColumn(
        "price_sk",
        sha2(concat_ws("||", col("series_bk"), col("effective_from").cast("string")), 256)
    ).withColumn(
        "effective_to", lead("effective_from", 1).over(w_ordered)
    ).withColumn(
        "is_current", col("effective_to").isNull()
    ).withColumn(
        "_source_system", lit("EIA_petroleum_prices")
    ).withColumn(
        "_ingested_at", current_timestamp()
    ).withColumn(
        "_updated_at", lit(None).cast("timestamp")
    ).select(
        "price_sk", "series_bk", "product_name", "units", "price",
        "effective_from", "effective_to", "is_current",
        "_source_system", "_ingested_at", "_updated_at"
    )

    target_table_obj = DeltaTable.forName(spark, silver_full_name)

    step1_result = (target_table_obj.alias("t")
        .merge(final_df.alias("s"), "t.series_bk = s.series_bk AND t.is_current = true")
        .whenMatchedUpdate(
            condition="t.price <> s.price",
            set={
                "effective_to": "s.effective_from",
                "is_current": "false",
                "_updated_at": "current_timestamp()"
            }
        )
        .execute()
    )
    step1_stats = step1_result.collect()[0].asDict()
    logger.info(f"MERGE step 1 (close current) completed: {step1_stats}")

    step2_result = (target_table_obj.alias("t")
        .merge(final_df.alias("s"), "t.price_sk = s.price_sk")
        .whenNotMatchedInsert(
            values={
                "price_sk": "s.price_sk",
                "series_bk": "s.series_bk",
                "product_name": "s.product_name",
                "units": "s.units",
                "price": "s.price",
                "effective_from": "s.effective_from",
                "effective_to": "s.effective_to",
                "is_current": "s.is_current",
                "_source_system": "s._source_system",
                "_ingested_at": "s._ingested_at",
                "_updated_at": "s._updated_at"
            }
        )
        .execute()
    )
    step2_stats = step2_result.collect()[0].asDict()
    logger.info(f"MERGE step 2 (insert new version) completed: {step2_stats}")

    if step1_stats["num_affected_rows"] == 0 and step2_stats["num_affected_rows"] == 0 and bronze_df.isEmpty():
        raise ValueError(f"Source table {bronze_full_name} is empty, aborting pipeline")

except Exception as e:
    logger.error(f"Pipeline failed: {e}")
    raise